# 11 - Images for the code of Shalabi et al.

Check of our training pipeline: the published code of Shalabi et al. (2026) is run, with minimal fixes, on our
images. If it reaches an AUC close to ours (about 0.92), the gap with the AUC of 0.9999 of the paper does not come
from our training code.

Their code reads one 8-bit grey-level PNG per component, `GoodQuality/GQ_HNE_<index>.png`,
`BadQuality/BQ_HNZ_<index>.png` and so on, and draws 22,250 records per class itself. This notebook writes the CV
images of notebook 02d (the images of the replication run) in that layout, for all the records with data:

- the u, v and w components become the HNE, HNN and HNZ files (whatever the real channel code);
- grey levels in [0, 1] written as round(255 x value), highest frequency in the top row;
- the export index is the feature index of the record; `exported_records.csv` maps it to the record.

The images are written to `EXPORT_DIR` (about 300,000 small files); their code is then run from its own folder,
outside this repository, with `--data-dir` pointing to `EXPORT_DIR`.

## 0. Setup

In [1]:
from dataclasses import replace
from pathlib import Path

import h5py
import numpy as np
from PIL import Image

from wqu.config import CV_GOOD_SOURCE, OBSPY_REPRESENTATION, load_config
from wqu.preprocessing.features import representation_feature_path
from wqu.preprocessing.pipeline import read_feature_records
from wqu.preprocessing.png_export import (
    EXPORT_INDEX_COLUMN,
    ClassFolder,
    export_png_images,
    figure_orientation,
    png_path,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
# Layout read by prepare_dataset in the dataset.py of Shalabi et al.
CLASS_FOLDERS = {0: ClassFolder(folder="GoodQuality", prefix="GQ"), 1: ClassFolder(folder="BadQuality", prefix="BQ")}
CHANNELS = ("HNE", "HNN", "HNZ")
BATCH_SIZE = 1024

config = load_config(CONFIG_PATH)
label_column = config.columns.label
EXPORT_DIR = config.preprocessing.output_dir.parent / "shalabi_png" / "Pipeline"

cv_settings = replace(config.preprocessing.obspy_spectrogram, good_source=CV_GOOD_SOURCE)
features_path = representation_feature_path(replace(config.preprocessing, obspy_spectrogram=cv_settings),
                                            config.training.feature_source, config.training.representation)
records = read_feature_records(features_path, config.columns.text)
print(f"Feature file: {features_path}")
print(f"Records: {len(records):,}; per label: {records[label_column].value_counts().sort_index().to_dict()}")
print(f"Export folder: {EXPORT_DIR}")

/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature file: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/processed/seisbench/obspy/original_recorded_linear_decibel_min_max_viridis_luminance_w128o0.9p8_64x64/features.h5
Records: 100,093; per label: {0: 64213, 1: 35880}
Export folder: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/shalabi_png/Pipeline


## 1. Export

In [2]:
exported = export_png_images(features_path, records, label_column, CLASS_FOLDERS, CHANNELS, EXPORT_DIR, BATCH_SIZE)
for label, class_folder in CLASS_FOLDERS.items():
    n_files = sum(1 for _ in (EXPORT_DIR / class_folder.folder).glob("*.png"))
    print(f"{class_folder.folder}: {(exported[label_column] == label).sum():,} records, {n_files:,} PNG files")

GoodQuality: 64,213 records, 192,639 PNG files
BadQuality: 35,880 records, 107,640 PNG files


## 2. Check

The PNG of a few records read back as the code of Shalabi et al. reads them (`convert('L')`, values divided by 255)
must match the stored images within the 8-bit quantization (at most 0.5 / 255).

In [3]:
CHECKED_RECORDS = 20
QUANTIZATION_TOLERANCE = 0.5 / 255 + 1e-6

rng = np.random.default_rng(config.training.random_seed)
checked = exported.iloc[rng.choice(len(exported), size=CHECKED_RECORDS, replace=False)]
largest_difference = 0.0
with h5py.File(features_path, mode="r") as stored:
    for export_index, label in zip(checked[EXPORT_INDEX_COLUMN], checked[label_column]):
        image = stored[OBSPY_REPRESENTATION][int(export_index)].astype(np.float64)
        for component, channel in enumerate(CHANNELS):
            path = png_path(EXPORT_DIR, CLASS_FOLDERS[int(label)], channel, int(export_index))
            pixels = np.asarray(Image.open(path).convert("L"), dtype=np.float64) / 255
            largest_difference = max(largest_difference, float(np.abs(figure_orientation(pixels) - image[component]).max()))
print(f"Largest difference over {CHECKED_RECORDS} records: {largest_difference:.5f} "
      f"(tolerance {QUANTIZATION_TOLERANCE:.5f}): {'ok' if largest_difference <= QUANTIZATION_TOLERANCE else 'TOO LARGE'}")

Largest difference over 20 records: 0.00196 (tolerance 0.00196): ok
